[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/090.%20The%20Global%20Sourcing%20Offshoring%20vs.%20Reshoring%20Problem/P90-Tier-1_executed.ipynb)

---

# 90. The Global Sourcing: Offshoring vs. Reshoring Problem
## Tier 1: Mathematical Formulation and Numerical Solution

### Purpose and scope

This notebook allocates the monthly demand for three components across three sourcing locations. It formulates a weighted objective that combines purchasing and fixed costs, a risk penalty, and a quality benefit. Demand, capacity, and risk-diversification requirements restrict the feasible allocations.

The notebook follows a continuous, constrained optimization workflow using SciPy's SLSQP solver. It also examines how the allocation responds to changes in the objective weights. The resulting numerical solution is a candidate found by the solver; the method does not establish global optimality.

### Model assumptions

- Each product's monthly demand must be met exactly, and sourcing quantities are continuous and nonnegative.
- Each location has a shared monthly capacity across all products.
- A location's fixed cost is charged once if any positive quantity is sourced from that location.
- A product may receive at most the specified fraction of its demand from locations classified as high risk.
- The model uses one monthly planning period. Lead times are provided as data but do not affect the objective or constraints.
- The model does not represent supplier-level choices, integer shipment sizes, uncertainty, inventory, or time-dependent decisions.

### The example

GlobalTech Industries sources microprocessors, displays, and batteries from China, Mexico, and the United States. Monthly demand is 10,000, 8,000, and 12,000 units, respectively. Unit costs, risk factors, quality scores, lead times, and capacities are specified in the data cell below.

The word *reshoring* describes the broader sourcing context; the modeled alternatives here are the three locations listed in the data. There is no separate hybrid-location decision.

### Workflow

1. Define the products, locations, parameters, and decision variables.
2. Evaluate the weighted objective: cost plus risk penalty minus quality benefit.
3. impose demand, capacity, risk-diversification, and nonnegativity requirements.
4. Use SLSQP to search for a feasible allocation with a low objective value.
5. Inspect the allocation, objective components, visualizations, and weight-sensitivity results.

### Interpreting the results

The allocation table reports quantities by product and location. The cost breakdown separates purchasing cost from fixed cost. Risk and quality totals are weighted in the objective, but they are not monetary costs. Because the weights combine quantities with different units and scales, the objective is a weighted score, not a dollar amount. Objective values from different weight scenarios are not directly comparable as measures of economic cost.

The fixed-charge term changes discontinuously when a location's allocated quantity moves from zero to a positive value. SLSQP is a continuous constrained optimizer and does not model binary activation decisions. Consequently, its success flag indicates termination according to the solver's criteria, not proof of a global optimum; the fixed-charge discontinuity also makes the numerical search less reliable near location-activation boundaries.

## Mathematical model

### Decision variables and sets

Let $i \in I$ index products and $j \in J$ index sourcing locations. The decision variable $x_{ij} \geq 0$ is the quantity of product $i$ sourced from location $j$ during the monthly planning period. In the code, the variables are flattened for the solver and reshaped into an array with one period.

Let $d_i$ be demand for product $i$, $K_j$ the capacity of location $j$, $c_j$ its unit cost, $F_j$ its fixed cost, $r_j$ its risk factor, and $q_j$ its quality score. The weights $\alpha$, $\beta$, and $\gamma$ determine the relative contributions of cost, risk, and quality.

### Weighted objective

The model minimizes a scalar score combining the three criteria:

$$\min_{x \geq 0}\quad
\alpha\left(\sum_{i \in I}\sum_{j \in J} c_j x_{ij}
+ \sum_{j \in J} F_j\,\mathbf{1}\!\left[\sum_{i \in I}x_{ij}>0\right]\right)
+ \beta\sum_{i \in I}\sum_{j \in J} r_j x_{ij}
- \gamma\sum_{i \in I}\sum_{j \in J} q_j x_{ij}.$$

Here, $\mathbf{1}[\cdot]$ equals 1 when its condition is true and 0 otherwise. The fixed cost is therefore charged once per active location, even when that location supplies multiple products. The negative sign on the quality term makes a higher quality score reduce the minimized objective. In the code, `objective_function` computes these terms. The weights combine values with different units and scales, so the resulting score should not be interpreted as dollars.

### Constraints

Every product's demand must be met exactly:

$$\sum_{j \in J}x_{ij}=d_i \qquad \forall i \in I.$$

The total quantity assigned to a location cannot exceed its capacity:

$$\sum_{i \in I}x_{ij}\leq K_j \qquad \forall j \in J.$$

For each product, no more than the allowed share of demand may come from high-risk locations. Define $H=\{j \in J:r_j\geq \tau\}$, where $\tau$ is the high-risk threshold, and let $\rho$ be the maximum high-risk share. Then:

$$\sum_{j \in H}x_{ij}\leq \rho d_i \qquad \forall i \in I.$$

The code expresses the same risk limit as $\rho\sum_j x_{ij}-\sum_{j\in H}x_{ij}\geq 0$; the demand equality makes the two forms equivalent. `constraint_demands`, `constraint_capacities`, and `constraint_risk_diversification` evaluate these requirements. Nonnegativity is enforced through the solver bounds.

### Solver interpretation and limitations

SLSQP searches over continuous decision variables while respecting equality and inequality constraints. The demand and capacity constraints are linear, and the risk constraint is linear for this data. However, the fixed-charge indicator makes the objective discontinuous at zero allocation. The resulting model is not a smooth convex optimization problem, and this SLSQP implementation does not guarantee a globally optimal solution. A reported feasible candidate should therefore be interpreted as a numerical result from the specified method, not as a proof of optimality.

In [ ]:
# Import the numerical, tabular, plotting, and optimization tools used below.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
class SourcingData:
    """Parameters for the single-period global sourcing example."""

    def __init__(self):
        self.products = ['Microprocessor', 'Display', 'Battery']
        self.demands = {
            'Microprocessor': 10000,
            'Display': 8000,
            'Battery': 12000,
        }

        self.locations = ['China', 'Mexico', 'US']
        self.unit_costs = {'China': 25, 'Mexico': 35, 'US': 50}
        self.risk_factors = {'China': 0.8, 'Mexico': 0.4, 'US': 0.1}
        self.quality_scores = {'China': 0.7, 'Mexico': 0.8, 'US': 0.95}
        self.lead_times = {'China': 8, 'Mexico': 3, 'US': 1}  # Weeks
        self.capacities = {'China': 50000, 'Mexico': 30000, 'US': 25000}

        # Each fixed cost is charged once when its location supplies any product.
        self.fixed_costs = {'China': 10000, 'Mexico': 8000, 'US': 12000}

        self.alpha = 0.6  # Cost weight
        self.beta = 0.3   # Risk weight
        self.gamma = 0.1  # Quality weight

        self.high_risk_threshold = 0.6
        self.max_high_risk_percentage = 0.6


data = SourcingData()
print(f'Products: {data.products}')
print(f'Sourcing locations: {data.locations}')
print(f'Total monthly demand: {sum(data.demands.values()):,} units')

In [ ]:
def objective_function(x_flat, data):
    """Return the weighted cost, risk, and quality score for an allocation."""
    n_products = len(data.products)
    n_locations = len(data.locations)
    x = x_flat.reshape(n_products, n_locations, 1)

    total_variable_cost = 0.0
    total_fixed_cost = 0.0
    risk_penalty = 0.0
    quality_benefit = 0.0

    for i, product in enumerate(data.products):
        for j, location in enumerate(data.locations):
            quantity = x[i, j, 0]
            total_variable_cost += data.unit_costs[location] * quantity
            risk_penalty += data.risk_factors[location] * quantity
            quality_benefit += data.quality_scores[location] * quantity

    # A location-level fixed charge is counted once, regardless of its product mix.
    for j, location in enumerate(data.locations):
        location_quantity = np.sum(x[:, j, 0])
        if location_quantity > 0:
            total_fixed_cost += data.fixed_costs[location]

    return (
        data.alpha * (total_variable_cost + total_fixed_cost)
        + data.beta * risk_penalty
        - data.gamma * quality_benefit
    )

In [ ]:
def constraint_demands(x_flat, data):
    """Return demand residuals; equality constraints require each to be zero."""
    n_products = len(data.products)
    n_locations = len(data.locations)
    x = x_flat.reshape(n_products, n_locations, 1)

    residuals = []
    for i, product in enumerate(data.products):
        total_sourced = np.sum(x[i, :, 0])
        residuals.append(total_sourced - data.demands[product])

    return residuals


def constraint_capacities(x_flat, data):
    """Return remaining capacity; inequality constraints require nonnegative values."""
    n_products = len(data.products)
    n_locations = len(data.locations)
    x = x_flat.reshape(n_products, n_locations, 1)

    residuals = []
    for j, location in enumerate(data.locations):
        total_used = np.sum(x[:, j, 0])
        residuals.append(data.capacities[location] - total_used)

    return residuals


def constraint_risk_diversification(x_flat, data):
    """Return the permitted minus actual high-risk quantity for each product."""
    n_products = len(data.products)
    n_locations = len(data.locations)
    x = x_flat.reshape(n_products, n_locations, 1)
    high_risk_indices = [
        j
        for j, location in enumerate(data.locations)
        if data.risk_factors[location] >= data.high_risk_threshold
    ]

    residuals = []
    if high_risk_indices:
        for i, product in enumerate(data.products):
            total_sourced = np.sum(x[i, :, 0])
            high_risk_total = np.sum(x[i, high_risk_indices, 0])
            maximum_allowed = data.max_high_risk_percentage * total_sourced
            residuals.append(maximum_allowed - high_risk_total)

    return residuals

## Data and decision-variable layout

The class stores the example parameters in dictionaries keyed by product or location. This makes the mapping between a named parameter and its numerical value explicit. The decision array has one quantity for every product-location pair; it is flattened only because the solver interface expects a one-dimensional vector. The objective and constraint functions reshape that vector to keep the product and location indices visible in their calculations.

Capacity is shared across products at each location. A location's lead time is retained as descriptive data, but it is not used in this single-period model. The high-risk threshold classifies locations using their risk factors, and the risk-diversification constraint is applied separately to each product.

In [ ]:
# Set dimensions in the same product-by-location order used by the model functions.
n_products = len(data.products)
n_locations = len(data.locations)
n_variables = n_products * n_locations

# The starting point distributes each product's demand in proportion to location capacity.
total_capacity = sum(data.capacities.values())
x0 = np.zeros(n_variables)
for i, product in enumerate(data.products):
    demand = data.demands[product]
    for j, location in enumerate(data.locations):
        proportion = data.capacities[location] / total_capacity
        x0[i * n_locations + j] = demand * proportion

# Bounds enforce nonnegative sourcing quantities.
bounds = [(0, None) for _ in range(n_variables)]

# SLSQP expects equality functions to return zero and inequality functions to be nonnegative.
constraints = []
for i in range(n_products):
    def demand_constraint(x_flat, idx=i):
        return constraint_demands(x_flat, data)[idx]

    constraints.append({'type': 'eq', 'fun': demand_constraint})

for j in range(n_locations):
    def capacity_constraint(x_flat, idx=j):
        return constraint_capacities(x_flat, data)[idx]

    constraints.append({'type': 'ineq', 'fun': capacity_constraint})

n_risk_constraints = len(constraint_risk_diversification(x0, data))
for k in range(n_risk_constraints):
    def risk_constraint(x_flat, idx=k):
        return constraint_risk_diversification(x_flat, data)[idx]

    constraints.append({'type': 'ineq', 'fun': risk_constraint})

print(f'Number of decision variables: {n_variables}')
print(f'Number of constraints: {len(constraints)}')
print(f'Initial total allocation: {np.sum(x0):,.0f} units')

## Solve the constrained optimization problem

The initial allocation gives the solver a starting point; it is not assumed to be optimal. SLSQP adjusts the continuous quantities to reduce the weighted objective while satisfying the supplied equality constraints, inequality constraints, and bounds. Since the objective contains fixed charges that switch on at positive allocation, the solver's result must be checked for feasibility and interpreted with the limitations described in the mathematical model.

In [ ]:
print('Solving the global sourcing optimization problem...')

result = minimize(
    objective_function,
    x0,
    args=(data,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-9},
)

solution_valid = False
if result.success:
    demand_residuals = np.asarray(constraint_demands(result.x, data))
    capacity_residuals = np.asarray(constraint_capacities(result.x, data))
    risk_residuals = np.asarray(constraint_risk_diversification(result.x, data))
    bounds_valid = np.all(result.x >= -1e-6)
    solution_valid = (
        np.all(np.abs(demand_residuals) <= 1e-4)
        and np.all(capacity_residuals >= -1e-4)
        and np.all(risk_residuals >= -1e-4)
        and bounds_valid
    )

    if solution_valid:
        print('Solver terminated successfully and the candidate satisfies the model constraints.')
        print(f'Weighted objective score: {result.fun:,.4f}')
        print(f'Iterations: {result.nit}')
        print(f'Function evaluations: {result.nfev}')
        print('This numerical result is not a proof of global optimality.')
    else:
        print('The solver terminated successfully, but its candidate failed the feasibility checks.')
        print(f'Solver message: {result.message}')
else:
    print('Optimization did not terminate successfully.')
    print(f'Solver message: {result.message}')

## Inspect the candidate allocation

The next cell reports results only when the solver has terminated successfully and the candidate passes explicit checks for demand, capacity, risk diversification, and nonnegativity. The allocation table is rounded for display; cost and objective calculations use the unrounded solver values. Fixed costs are reported once for each active location, consistent with the objective function.

In [ ]:
if solution_valid:
    optimal_allocation = result.x.reshape(n_products, n_locations, 1)
    allocation_matrix = optimal_allocation[:, :, 0]

    results_df = pd.DataFrame(
        allocation_matrix.round(0),
        index=data.products,
        columns=data.locations,
    )
    results_df['Total Demand'] = [data.demands[product] for product in data.products]
    total_allocation = pd.Series(
        allocation_matrix.sum(axis=0), index=data.locations
    )
    results_df.loc['Total Allocation', data.locations] = total_allocation.round(0)
    results_df.loc['Total Allocation', 'Total Demand'] = sum(data.demands.values())

    print('=== SOURCING ALLOCATION (UNITS) ===')
    print(results_df.to_string())

    total_variable_cost = 0.0
    total_fixed_cost = 0.0
    total_risk_penalty = 0.0
    total_quality_benefit = 0.0

    print('\n=== LOCATION-LEVEL BREAKDOWN ===')
    for j, location in enumerate(data.locations):
        location_quantity = np.sum(allocation_matrix[:, j])
        if location_quantity > 0:
            variable_cost = data.unit_costs[location] * location_quantity
            fixed_cost = data.fixed_costs[location]
            risk_penalty = data.risk_factors[location] * location_quantity
            quality_benefit = data.quality_scores[location] * location_quantity

            total_variable_cost += variable_cost
            total_fixed_cost += fixed_cost
            total_risk_penalty += risk_penalty
            total_quality_benefit += quality_benefit

            print(f'{location}:')
            print(f'  Quantity: {location_quantity:,.2f} units')
            print(f'  Variable cost: ${variable_cost:,.2f}')
            print(f'  Fixed cost: ${fixed_cost:,.2f}')
            print(f'  Risk penalty (unweighted): {risk_penalty:,.2f}')
            print(f'  Quality benefit (unweighted): {quality_benefit:,.2f}')

    cost_component = data.alpha * (total_variable_cost + total_fixed_cost)
    risk_component = data.beta * total_risk_penalty
    quality_component = data.gamma * total_quality_benefit
    reconstructed_objective = cost_component + risk_component - quality_component

    print('\n=== WEIGHTED OBJECTIVE COMPONENTS ===')
    print(f'Total variable cost: ${total_variable_cost:,.2f}')
    print(f'Total fixed cost: ${total_fixed_cost:,.2f}')
    print(f'Total risk penalty (unweighted): {total_risk_penalty:,.2f}')
    print(f'Total quality benefit (unweighted): {total_quality_benefit:,.2f}')
    print(f'Weighted cost contribution: {cost_component:,.4f}')
    print(f'Weighted risk contribution: {risk_component:,.4f}')
    print(f'Weighted quality contribution: -{quality_component:,.4f}')
    print(f'Objective function value: {result.fun:,.4f}')
    print(
        'Objective reconstruction matches solver value: '
        f'{np.isclose(reconstructed_objective, result.fun, rtol=1e-7, atol=1e-5)}'
    )
else:
    print('A feasible solver candidate is not available for result analysis.')

## Visualize the candidate solution

The heatmap displays product-location quantities, the bar chart compares variable plus fixed cost by active location, and the scatter plot places each used location according to its risk factor and quality score. In the scatter plot, marker size represents sourced volume. The final panel shows each weighted objective contribution; the quality contribution is plotted as negative because quality is a benefit in a minimization objective.

In [ ]:
if solution_valid:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Global Sourcing Optimization Analysis', fontsize=16, fontweight='bold')

    ax1 = axes[0, 0]
    sns.heatmap(
        allocation_matrix,
        annot=True,
        fmt='.0f',
        xticklabels=data.locations,
        yticklabels=data.products,
        cmap='Blues',
        cbar_kws={'label': 'Sourced quantity (units)'},
        ax=ax1,
    )
    ax1.set_title('Sourcing Allocation')
    ax1.set_xlabel('Sourcing location')
    ax1.set_ylabel('Product')

    ax2 = axes[0, 1]
    location_costs = []
    location_names = []
    for j, location in enumerate(data.locations):
        quantity = np.sum(allocation_matrix[:, j])
        if quantity > 0:
            location_costs.append(
                data.unit_costs[location] * quantity + data.fixed_costs[location]
            )
            location_names.append(location)

    bars = ax2.bar(location_names, location_costs, color='#4C78A8')
    ax2.set_title('Purchasing and Fixed Cost by Location')
    ax2.set_xlabel('Sourcing location')
    ax2.set_ylabel('Cost ($)')
    ax2.tick_params(axis='x', rotation=45)
    if location_costs:
        for bar, cost in zip(bars, location_costs):
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f'${cost:,.0f}',
                ha='center',
                va='bottom',
            )

    ax3 = axes[1, 0]
    for j, location in enumerate(data.locations):
        quantity = np.sum(allocation_matrix[:, j])
        if quantity > 0:
            ax3.scatter(
                data.risk_factors[location],
                data.quality_scores[location],
                s=quantity / 100,
                alpha=0.7,
                label=location,
            )

    ax3.set_xlabel('Risk factor')
    ax3.set_ylabel('Quality score')
    ax3.set_title('Risk and Quality by Used Location')
    ax3.legend(title='Location')
    ax3.grid(True, alpha=0.3)
    ax3.text(
        0.02,
        0.02,
        'Marker size is proportional to sourced volume.',
        transform=ax3.transAxes,
        fontsize=9,
    )

    ax4 = axes[1, 1]
    components = ['Cost', 'Risk', 'Quality benefit']
    values = [cost_component, risk_component, -quality_component]
    colors = ['#4C78A8', '#F58518', '#54A24B']
    bars = ax4.bar(components, values, color=colors)
    ax4.set_title('Contributions to the Weighted Objective')
    ax4.set_xlabel('Objective component')
    ax4.set_ylabel('Weighted score contribution')
    ax4.axhline(y=0, color='black', linewidth=0.8, alpha=0.5)
    ax4.grid(True, alpha=0.3, axis='y')
    for bar, value in zip(bars, values):
        offset = 0.01 * max(abs(component) for component in values)
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            value + (offset if value >= 0 else -offset),
            f'{value:,.0f}',
            ha='center',
            va='bottom' if value >= 0 else 'top',
        )

    plt.tight_layout()
    plt.show()
else:
    print('A feasible solver candidate is not available for visualization.')

## Sensitivity analysis: objective weights

The sensitivity analysis repeats the same SLSQP model with different weights for cost, risk, and quality. It keeps the data, constraints, bounds, solver family, and starting point unchanged. The location percentages show how the sourcing mix responds to the changed trade-offs. Because each scenario defines a different weighted objective, its objective score should not be compared directly with scores from other scenarios as though they were the same cost measure. As with the main solve, solver success is not a guarantee of global optimality.

In [ ]:
if solution_valid:
    print('=== WEIGHT SENSITIVITY ANALYSIS ===')
    print('Each scenario changes the relative importance of cost, risk, and quality.')

    weight_scenarios = [
        {'name': 'Cost-focused', 'alpha': 0.8, 'beta': 0.15, 'gamma': 0.05},
        {'name': 'Balanced', 'alpha': 0.6, 'beta': 0.3, 'gamma': 0.1},
        {'name': 'Risk-averse', 'alpha': 0.4, 'beta': 0.5, 'gamma': 0.1},
        {'name': 'Quality-focused', 'alpha': 0.5, 'beta': 0.2, 'gamma': 0.3},
    ]

    sensitivity_results = []
    original_weights = (data.alpha, data.beta, data.gamma)

    try:
        for scenario in weight_scenarios:
            data.alpha = scenario['alpha']
            data.beta = scenario['beta']
            data.gamma = scenario['gamma']

            result_sensitivity = minimize(
                objective_function,
                x0,
                args=(data,),
                method='SLSQP',
                bounds=bounds,
                constraints=constraints,
                options={'maxiter': 500},
            )

            if result_sensitivity.success:
                allocation = result_sensitivity.x.reshape(
                    n_products, n_locations, 1
                )
                demand_ok = np.all(
                    np.abs(constraint_demands(result_sensitivity.x, data)) <= 1e-4
                )
                capacity_ok = np.all(
                    np.asarray(constraint_capacities(result_sensitivity.x, data)) >= -1e-4
                )
                risk_ok = np.all(
                    np.asarray(
                        constraint_risk_diversification(result_sensitivity.x, data)
                    ) >= -1e-4
                )
                bounds_ok = np.all(result_sensitivity.x >= -1e-6)

                if demand_ok and capacity_ok and risk_ok and bounds_ok:
                    total_quantity = np.sum(allocation)
                    location_percentages = (
                        np.sum(allocation[:, :, 0], axis=0) / total_quantity * 100
                    )
                    sensitivity_results.append(
                        {
                            'Scenario': scenario['name'],
                            'Objective score': result_sensitivity.fun,
                            'China (%)': location_percentages[0],
                            'Mexico (%)': location_percentages[1],
                            'US (%)': location_percentages[2],
                        }
                    )
                else:
                    print(
                        f"Scenario {scenario['name']} terminated successfully "
                        'but failed a feasibility check.'
                    )
            else:
                print(
                    f"Scenario {scenario['name']} did not terminate successfully: "
                    f'{result_sensitivity.message}'
                )
    finally:
        # Restore the main-example weights so later calculations retain their meaning.
        data.alpha, data.beta, data.gamma = original_weights

    if sensitivity_results:
        sensitivity_df = pd.DataFrame(sensitivity_results)
        print('\nFeasible sensitivity results:')
        print(sensitivity_df.round(3).to_string(index=False))

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

        sensitivity_df.set_index('Scenario')[
            ['China (%)', 'Mexico (%)', 'US (%)']
        ].plot(
            kind='bar',
            stacked=True,
            ax=ax1,
            color=['#4C78A8', '#F58518', '#54A24B'],
        )
        ax1.set_title('Allocation by Weight Scenario')
        ax1.set_xlabel('Weight scenario')
        ax1.set_ylabel('Share of total sourced quantity (%)')
        ax1.legend(title='Location')
        ax1.tick_params(axis='x', rotation=45)

        ax2.bar(
            sensitivity_df['Scenario'],
            sensitivity_df['Objective score'],
            color='#72B7B2',
        )
        ax2.set_title('Weighted Objective Score by Scenario')
        ax2.set_xlabel('Weight scenario')
        ax2.set_ylabel('Weighted objective score')
        ax2.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()
    else:
        print('No sensitivity scenario produced a feasible solver candidate.')
else:
    print('A feasible main candidate is required before sensitivity analysis.')